[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso/02-clasificacion-texto.ipynb)

# 02 — Clasificación de texto

> **Bloque:** Casos de uso | **Nivel:** Práctico
>
> Notebook complementario al tutorial [02-clasificacion-texto.md](../../casos-de-uso/02-clasificacion-texto.md)

Experimentamos con los tres enfoques de clasificación: zero-shot, few-shot y ML clásico.

In [ ]:
# %pip install anthropic scikit-learn pandas matplotlib python-dotenv

import anthropic
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('✓ Listo')

## Dataset de ejemplo

Usaremos reseñas de productos para análisis de sentimiento.

In [ ]:
dataset = [
    {'texto': 'Producto increíble, superó todas mis expectativas. Lo recomiendo al 100%.', 'etiqueta': 'POSITIVO'},
    {'texto': 'Llegó roto y el servicio de atención al cliente no resolvió nada.', 'etiqueta': 'NEGATIVO'},
    {'texto': 'El artículo tiene las dimensiones indicadas en la descripción.', 'etiqueta': 'NEUTRO'},
    {'texto': 'Muy buena relación calidad-precio. Repetiré la compra.', 'etiqueta': 'POSITIVO'},
    {'texto': 'No funciona como se anuncia. Decepcionante.', 'etiqueta': 'NEGATIVO'},
    {'texto': 'Envío en el plazo habitual. Sin incidencias.', 'etiqueta': 'NEUTRO'},
    {'texto': 'Calidad excelente, materiales de primera. Muy contento.', 'etiqueta': 'POSITIVO'},
    {'texto': 'Tardó el doble de lo prometido y llegó con embalaje dañado.', 'etiqueta': 'NEGATIVO'},
    {'texto': 'Cumple su función básica sin destacar.', 'etiqueta': 'NEUTRO'},
    {'texto': 'Fantástico producto. Ya es la tercera vez que lo compro.', 'etiqueta': 'POSITIVO'},
    {'texto': 'Me cobraron dos veces y nadie me responde.', 'etiqueta': 'NEGATIVO'},
    {'texto': 'El color es ligeramente diferente al de la foto.', 'etiqueta': 'NEUTRO'},
]

df = pd.DataFrame(dataset)
print(df['etiqueta'].value_counts())
df.head()

## 1. Clasificación Zero-shot con LLM

In [ ]:
def clasificar_zeroshot(texto: str) -> dict:
    prompt = f"""Clasifica el sentimiento en POSITIVO, NEGATIVO o NEUTRO.
Responde SOLO con JSON: {{"etiqueta": "...", "confianza": 0.0}}

Texto: "{texto}""""
    r = client.messages.create(
        model=MODELO, max_tokens=60, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    try:
        return json.loads(r.content[0].text)
    except:
        return {'etiqueta': 'ERROR', 'confianza': 0.0}

# Clasificar todos los ejemplos
resultados_zs = []
for row in dataset:
    res = clasificar_zeroshot(row['texto'])
    resultados_zs.append({
        'texto': row['texto'][:50] + '...',
        'real': row['etiqueta'],
        'predicho': res['etiqueta'],
        'confianza': res.get('confianza', 0),
        'correcto': row['etiqueta'] == res['etiqueta']
    })
    time.sleep(0.3)

df_zs = pd.DataFrame(resultados_zs)
accuracy = df_zs['correcto'].mean()
print(f'\nAccuracy zero-shot: {accuracy:.1%}')
df_zs[['real', 'predicho', 'correcto', 'confianza']]

## 2. Clasificación Few-shot con LLM

In [ ]:
EJEMPLOS = [
    ('Producto perfecto, muy recomendable', 'POSITIVO'),
    ('Pésima calidad, no lo compréis', 'NEGATIVO'),
    ('El tamaño es el indicado en la ficha', 'NEUTRO'),
]

def clasificar_fewshot(texto: str) -> dict:
    ejemplos_str = '\n'.join([f'Texto: "{t}" → {e}' for t, e in EJEMPLOS])
    prompt = f"""Clasifica el sentimiento en POSITIVO, NEGATIVO o NEUTRO.

Ejemplos:
{ejemplos_str}

Clasifica este texto. Responde SOLO con JSON: {{"etiqueta": "...", "confianza": 0.0}}
Texto: "{texto}""""
    r = client.messages.create(
        model=MODELO, max_tokens=60, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    try:
        return json.loads(r.content[0].text)
    except:
        return {'etiqueta': 'ERROR', 'confianza': 0.0}

resultados_fs = []
for row in dataset:
    res = clasificar_fewshot(row['texto'])
    resultados_fs.append({
        'real': row['etiqueta'],
        'predicho': res['etiqueta'],
        'correcto': row['etiqueta'] == res['etiqueta']
    })
    time.sleep(0.3)

df_fs = pd.DataFrame(resultados_fs)
print(f'Accuracy few-shot: {df_fs["correcto"].mean():.1%}')

## 3. Clasificación con ML clásico (TF-IDF + Regresión Logística)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Dataset ampliado para ML (necesita más ejemplos)
textos_ml = [d['texto'] for d in dataset] * 5  # Replicar para demostración
etiquetas_ml = [d['etiqueta'] for d in dataset] * 5

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Validación cruzada
scores = cross_val_score(pipeline, textos_ml, etiquetas_ml, cv=3, scoring='accuracy')
print(f'Accuracy ML (cross-val): {scores.mean():.1%} ± {scores.std():.1%}')

# Entrenar con todo el dataset
pipeline.fit(textos_ml, etiquetas_ml)
print('\nModelo entrenado ✓')

## 4. Comparativa visual de los tres enfoques

In [ ]:
# Calcular accuracy ML sobre el dataset base
pred_ml = pipeline.predict([d['texto'] for d in dataset])
acc_ml = np.mean([p == d['etiqueta'] for p, d in zip(pred_ml, dataset)])

enfoques = ['Zero-shot\n(LLM)', 'Few-shot\n(LLM)', 'TF-IDF +\nLog. Reg.']
accuracies = [
    df_zs['correcto'].mean(),
    df_fs['correcto'].mean(),
    acc_ml
]
colores = ['#3498DB', '#2ECC71', '#E74C3C']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(enfoques, accuracies, color=colores, alpha=0.85, edgecolor='black', width=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Comparativa de enfoques de clasificación de texto')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)
ax.bar_label(bars, fmt='{:.0%}', padding=3, fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Prueba con tus propios textos

In [ ]:
mis_textos = [
    'Este producto me ha cambiado la vida',
    'Horrible, jamás volvería a comprarlo',
    'El producto tiene un peso de 500 gramos',
    # Añade más textos aquí
]

print(f'{"Texto":<45} {"LLM":^12} {"ML":^12}')
print('-' * 70)
for texto in mis_textos:
    llm = clasificar_zeroshot(texto)['etiqueta']
    ml  = pipeline.predict([texto])[0]
    print(f'{texto[:44]:<45} {llm:^12} {ml:^12}')
    time.sleep(0.3)

---
**Tutorial relacionado:** [02-clasificacion-texto.md](../../casos-de-uso/02-clasificacion-texto.md)